In [2]:
# ================================================================
# 🚨 MANEJO DE ERRORES PIPELINE - SILVER
# ================================================================
import traceback
from datetime import datetime
from zoneinfo import ZoneInfo
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, date_format


CATALOG = "Prueba Tecnica Dataknow"
SILVER_LH = "LH_SILVER_PRUEBA"
SCHEMA = "dbo"

PIPELINE_NAME = "SILVER"
BOGOTA_TZ = ZoneInfo("America/Bogota")

pipeline_errors = []

def ejecutar_con_manejo_errores(nombre_tarea, funcion):
    try:
        funcion()

        pipeline_errors.append(Row(
            pipeline=PIPELINE_NAME,
            tarea=nombre_tarea,
            estado="OK",
            mensaje="Ejecutado correctamente",
            detalle_error="",
            timestamp=datetime.now(BOGOTA_TZ).strftime("%Y-%m-%d %H:%M:%S")
        ))

        logger.info(f"✅ {nombre_tarea}")

    except Exception as e:
        pipeline_errors.append(Row(
            pipeline=PIPELINE_NAME,
            tarea=nombre_tarea,
            estado="ERROR",
            mensaje=str(e)[:500],
            detalle_error=traceback.format_exc()[:2000],
            timestamp=datetime.now(BOGOTA_TZ).strftime("%Y-%m-%d %H:%M:%S")
        ))

        logger.error(f"⚠️ Error en {nombre_tarea}: {str(e)[:200]}")
        logger.error("→ Registrado. Continuando...")

def guardar_errores_pipeline():
    if pipeline_errors:
        df = spark.createDataFrame(pipeline_errors)

        df = df.withColumn(
            "_fec_registro",
            date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss")
        )

        df.write.format("delta") \
          .mode("append") \
          .option("mergeSchema", "true") \
          .saveAsTable(f"`{CATALOG}`.{SILVER_LH}.{SILVER_SCHEMA}.pipeline_error_log")

        ok = len([e for e in pipeline_errors if e.estado == "OK"])
        err = len([e for e in pipeline_errors if e.estado == "ERROR"])

        logger.info(f"\n📋 pipeline_error_log SILVER: {ok} OK, {err} errores")

StatementMeta(, 53313e9a-d035-4285-982c-d89b1beaa476, 4, Finished, Available, Finished, False)

## Script Final

In [3]:
#   SILVER - LIMPIEZA Y CONFORMACIÓN                                   ║
#   Lee: BRZ_* (dato crudo con auditoría)                              ║
#   Crea: SLV_* (dato limpio) + ERR_* (errores documentados)           ║
# ║                                                                     ║
#  Cumple:                                                             ║
# ║   1. Eliminar duplicados (detecta anomalía 1 de Fase 1)             ║
# ║   2. Eliminar nulos corrompidos                                      ║
# ║   3. Detectar fechas fuera de rango (detecta anomalía 2 de Fase 1)  ║
# ║   4. Corregir inconsistencias (detecta anomalía 3 de Fase 1)        ║
# ║   5. Estandarizar tipos de datos                                     ║
# ║   6. Validar integridad referencial → ERR_*                         ║
# ║   7. Estrategia documentada de nulos                                 ║
# ║   8. Enmascaramiento PII (hash + masking) 
# ║   9. Detección de transacciones sospechosas (ind_sospechoso)         ║
# ║  10. Reporte de calidad de datos                                     ║
# ╚══════════════════════════════════════════════════════════════════════╝

from pyspark.sql.functions import (
    col, count, lit, when, current_timestamp, sha2, concat_ws,
    trim, upper, round as spark_round, sum as spark_sum,
    row_number, coalesce
)
from pyspark.sql import Window
from pyspark.sql.types import (
    StringType, IntegerType, LongType, DoubleType, DateType, BooleanType
)
from datetime import datetime
import logging
import time

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("SilverPipeline")

# ---------------------------
# Configuración
# ---------------------------
from pyspark.sql.functions import (
    col, count, lit, when, current_timestamp, sha2, concat_ws, avg, stddev, to_utc_timestamp, date_format,
    trim, upper, round as spark_round, sum as spark_sum,
    row_number, coalesce
)

from pyspark.sql import Window
from pyspark.sql.types import (
    StringType, IntegerType, LongType, DoubleType, DateType, BooleanType
)
from datetime import datetime
import logging
import time

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("SilverPipeline")

# ---------------------------
# Configuración de ubicación
# ---------------------------
CATALOG = "Prueba Tecnica Dataknow"

# Origen (Bronze)
BRONZE_LH = "LH_BRONZE_PRUEBA"
BRONZE_SCHEMA = "dbo"
BRONZE_PREFIX = "brz_"

# Destino (Silver) — 
SILVER_LH = "LH_SILVER_PRUEBA"      
SILVER_SCHEMA = "dbo"
SILVER_PREFIX = "slv_"
ERROR_PREFIX = "err_"

FECHA_MIN = "2024-03-01"
FECHA_MAX = "2026-03-31"

quality_report = []
error_log = []
anomalias_detectadas = []


def bronze(t):
    return f"`{CATALOG}`.{BRONZE_LH}.{BRONZE_SCHEMA}.{BRONZE_PREFIX}{t.lower()}"

def silver(t):
    return f"`{CATALOG}`.{SILVER_LH}.{SILVER_SCHEMA}.{SILVER_PREFIX}{t.lower()}"

def error_table(t):
    return f"`{CATALOG}`.{SILVER_LH}.{SILVER_SCHEMA}.{ERROR_PREFIX}{t.lower()}"

spark.conf.set("spark.sql.session.timeZone","America/Bogota")

# ---------------------------
# Configuración de TODAS las tablas
# ---------------------------
TABLES_CONFIG = {
    "TB_CLIENTES_CORE": {
        "primary_key": "id_cli",
        "date_cols": ["fec_alta"],
        "non_null_cols": ["id_cli", "nomb_cli", "apell_cli", "tip_doc", "num_doc"],
        "numeric_positive_cols": ["score_buro"],
    },
    "TB_PRODUCTOS_CAT": {
        "primary_key": "cod_prod",
        "date_cols": [],
        "non_null_cols": ["cod_prod", "desc_prod", "tip_prod"],
        "numeric_positive_cols": ["tasa_ea", "plazo_max_meses"],
    },
    "TB_MOV_FINANCIEROS": {
        "primary_key": "id_mov",
        "date_cols": ["fec_mov"],
        "non_null_cols": ["id_mov", "id_cli", "cod_prod", "vr_mov"],
        "numeric_positive_cols": ["vr_mov"],
    },
    "TB_OBLIGACIONES": {
        "primary_key": "id_oblig",
        "date_cols": [],
        "non_null_cols": ["id_oblig", "id_cli", "cod_prod", "vr_aprobado"],
        "numeric_positive_cols": ["vr_aprobado", "vr_desembolsado", "sdo_capital", "vr_cuota"],
    },
    "TB_SUCURSALES_RED": {
        "primary_key": "cod_suc",
        "date_cols": [],
        "non_null_cols": ["cod_suc", "nom_suc", "tip_punto"],
        "numeric_positive_cols": [],
    },
    "TB_COMISIONES_LOG": {
        "primary_key": "id_comision",
        "date_cols": ["fec_cobro"],
        "non_null_cols": ["id_comision", "id_cli", "cod_prod", "vr_comision"],
        "numeric_positive_cols": ["vr_comision"],
    },
}


# ================================================================
# PASO 1: ELIMINAR DUPLICADOS
# Requisito: "Eliminar registros duplicados exactos"
# Detecta ANOMALÍA 1 (Fase 1): transacciones duplicadas
# ================================================================
def eliminar_duplicados(df, table_name, pk):
    logger.info(f"\n  🔍 [1/8] DUPLICADOS por '{pk}'...")

    w = Window.partitionBy(pk).orderBy("_ingest_timestamp")
    df_ranked = df.withColumn("_dup_rank", row_number().over(w))

    df_duplicados = df_ranked.filter(col("_dup_rank") > 1).drop("_dup_rank")
    df_unicos = df_ranked.filter(col("_dup_rank") == 1).drop("_dup_rank")

    num_dup = df_duplicados.count()

    if num_dup > 0:
        logger.info(f"  🔴 {num_dup:,} duplicados eliminados")

        # Enviar a tabla de errores con motivo
        df_duplicados.drop("_dup_rank") \
            .withColumn("_error_tipo", lit("DUPLICADO")) \
            .withColumn("_error_motivo", lit(f"Registro duplicado por {pk}")) \
            .withColumn("_fec_deteccion", current_timestamp()) \
            .write.format("delta").mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable(error_table(table_name))

        error_log.append({
            "tabla": table_name, "tipo": "DUPLICADO",
            "motivo": f"PK duplicada: {pk}", "registros": num_dup
        })
        anomalias_detectadas.append({
            "anomalia": "ANOMALÍA 1 (Fase 1)",
            "descripcion": "Transacciones duplicadas",
            "tabla": table_name, "detectados": num_dup
        })
    else:
        logger.info(f"  ✅ Sin duplicados")

    return df_unicos


# ================================================================
# PASO 2: ELIMINAR NULOS EN CAMPOS OBLIGATORIOS
# Requisito: "Eliminar registros con campos obligatorios nulos
#             o corrompidos"
# ================================================================
def eliminar_nulos_criticos(df, table_name, non_null_cols):
    logger.info(f"  🔍 [2/8] NULOS en campos obligatorios: {non_null_cols}")

    cond_nulo = lit(False)
    nulos_detalle = []

    for c in non_null_cols:
        if c in df.columns:
            cnt = df.filter(col(c).isNull()).count()
            if cnt > 0:
                nulos_detalle.append(f"{c}={cnt}")
                cond_nulo = cond_nulo | col(c).isNull()

    if nulos_detalle:
        total_nulos = df.filter(cond_nulo).count()
        logger.info(f"  🔴 {total_nulos:,} registros con nulos críticos: {', '.join(nulos_detalle)}")

        df.filter(cond_nulo) \
            .withColumn("_error_tipo", lit("NULO_CAMPO_OBLIGATORIO")) \
            .withColumn("_error_motivo", lit(f"Nulos en: {', '.join(nulos_detalle)}")) \
            .withColumn("_fec_deteccion", current_timestamp()) \
            .write.format("delta").mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable(error_table(table_name))

        df = df.filter(~cond_nulo)

        error_log.append({
            "tabla": table_name, "tipo": "NULO_OBLIGATORIO",
            "motivo": f"Campos: {', '.join(nulos_detalle)}", "registros": total_nulos
        })
    else:
        logger.info(f"  ✅ Sin nulos en campos obligatorios")

    return df


# ================================================================
# PASO 3: DETECTAR FECHAS FUERA DE RANGO
# Requisito: "Eliminar registros corrompidos"
# Detecta ANOMALÍA 2 (Fase 1): fechas fuera de rango
# ================================================================
def detectar_fechas_anomalas(df, table_name, date_cols):
    if not date_cols:
        return df

    logger.info(f"  🔍 [3/8] FECHAS fuera de rango: {date_cols} ({FECHA_MIN} → {FECHA_MAX})")

    fecha_min = lit(FECHA_MIN).cast("date")
    fecha_max = lit(FECHA_MAX).cast("date")

    for date_col in date_cols:
        if date_col not in df.columns:
            continue

        cond_anomala = (
            col(date_col).isNotNull() & (
                (col(date_col) < fecha_min) |
                (col(date_col) > fecha_max)
            )
        )

        num_anomalos = df.filter(cond_anomala).count()

        if num_anomalos > 0:
            logger.info(f"  🔴 {date_col}: {num_anomalos:,} fechas fuera de rango")

            # Enviar a errores
            df.filter(cond_anomala) \
                .withColumn("_error_tipo", lit("FECHA_FUERA_RANGO")) \
                .withColumn("_error_motivo", lit(f"{date_col} fuera del rango {FECHA_MIN} a {FECHA_MAX}")) \
                .withColumn("_fec_deteccion", current_timestamp()) \
                .write.format("delta").mode("append") \
                .option("mergeSchema", "true") \
                .saveAsTable(error_table(table_name))

            # Marcar con flag + guardar original + anular
            df = df.withColumn(
                f"_flag_{date_col}_anomala",
                when(cond_anomala, lit(True)).otherwise(lit(False))
            ).withColumn(
                f"_{date_col}_original",
                when(cond_anomala, col(date_col))
            ).withColumn(
                date_col,
                when(cond_anomala, lit(None).cast("date")).otherwise(col(date_col))
            )

            error_log.append({
                "tabla": table_name, "tipo": "FECHA_FUERA_RANGO",
                "motivo": f"{date_col} fuera de rango", "registros": num_anomalos
            })
            anomalias_detectadas.append({
                "anomalia": "ANOMALÍA 2 (Fase 1)",
                "descripcion": f"Fechas fuera de rango en {date_col}",
                "tabla": table_name, "detectados": num_anomalos
            })
        else:
            logger.info(f"  ✅ {date_col}: todas en rango")

    return df


# ================================================================
# PASO 4: CORREGIR INCONSISTENCIAS LÓGICAS
# Detecta ANOMALÍA 3 (Fase 1): sdo_capital > vr_desembolsado
# ================================================================
def corregir_inconsistencias(df, table_name):
    if table_name != "TB_OBLIGACIONES":
        return df

    logger.info(f"  🔍 [4/8] INCONSISTENCIA: sdo_capital <= vr_desembolsado")

    cond = col("sdo_capital") > col("vr_desembolsado")
    num_incons = df.filter(cond).count()

    if num_incons > 0:
        logger.info(f"  🔴 {num_incons:,} registros con sdo_capital > vr_desembolsado")

        # Enviar a errores
        df.filter(cond) \
            .withColumn("_error_tipo", lit("INCONSISTENCIA_LOGICA")) \
            .withColumn("_error_motivo", lit("sdo_capital mayor que vr_desembolsado")) \
            .withColumn("_fec_deteccion", current_timestamp()) \
            .write.format("delta").mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable(error_table(table_name))

        # Marcar + guardar original + corregir
        df = df.withColumn(
            "_flag_saldo_inconsistente",
            when(cond, lit(True)).otherwise(lit(False))
        ).withColumn(
            "_sdo_capital_original",
            when(cond, col("sdo_capital"))
        ).withColumn(
            "sdo_capital",
            when(cond, col("vr_desembolsado")).otherwise(col("sdo_capital"))
        )

        error_log.append({
            "tabla": table_name, "tipo": "INCONSISTENCIA_LOGICA",
            "motivo": "sdo_capital > vr_desembolsado", "registros": num_incons
        })
        anomalias_detectadas.append({
            "anomalia": "ANOMALÍA 3 (Fase 1)",
            "descripcion": "sdo_capital mayor que vr_desembolsado",
            "tabla": table_name, "detectados": num_incons
        })
    else:
        logger.info(f"  ✅ Saldos consistentes")

    return df


# ================================================================
# PASO 5: ESTANDARIZAR TIPOS DE DATOS
# Requisito: "Estandarizar tipos de datos"
# ================================================================
TIPOS_ESTANDAR = {
    "TB_CLIENTES_CORE": {
        "id_cli": IntegerType(), "nomb_cli": StringType(),
        "apell_cli": StringType(), "tip_doc": StringType(),
        "num_doc": StringType(), "fec_nac": DateType(),
        "fec_alta": DateType(), "cod_segmento": StringType(),
        "score_buro": IntegerType(), "ciudad_res": StringType(),
        "depto_res": StringType(), "estado_cli": StringType(),
        "canal_adquis": StringType(),
    },
    "TB_PRODUCTOS_CAT": {
        "cod_prod": StringType(), "desc_prod": StringType(),
        "tip_prod": StringType(), "tasa_ea": DoubleType(),
        "plazo_max_meses": IntegerType(), "cuota_min": DoubleType(),
        "comision_admin": DoubleType(), "estado_prod": StringType(),
    },
    "TB_MOV_FINANCIEROS": {
        "id_mov": IntegerType(), "id_cli": IntegerType(),
        "cod_prod": StringType(), "num_cuenta": StringType(),
        "fec_mov": DateType(), "hra_mov": StringType(),
        "vr_mov": DoubleType(), "tip_mov": StringType(),
        "cod_canal": StringType(), "cod_ciudad": StringType(),
        "cod_estado_mov": StringType(), "id_dispositivo": StringType(),
    },
    "TB_OBLIGACIONES": {
        "id_oblig": IntegerType(), "id_cli": IntegerType(),
        "cod_prod": StringType(), "vr_aprobado": DoubleType(),
        "vr_desembolsado": DoubleType(), "sdo_capital": DoubleType(),
        "vr_cuota": DoubleType(), "fec_desembolso": DateType(),
        "fec_venc": DateType(), "dias_mora_act": IntegerType(),
        "num_cuotas_pend": IntegerType(), "calif_riesgo": StringType(),
    },
    "TB_SUCURSALES_RED": {
        "cod_suc": StringType(), "nom_suc": StringType(),
        "tip_punto": StringType(), "ciudad": StringType(),
        "depto": StringType(), "latitud": DoubleType(),
        "longitud": DoubleType(), "activo": BooleanType(),
    },
    "TB_COMISIONES_LOG": {
        "id_comision": IntegerType(), "id_cli": IntegerType(),
        "cod_prod": StringType(), "fec_cobro": DateType(),
        "vr_comision": DoubleType(), "tip_comision": StringType(),
        "estado_cobro": StringType(),
    },
}


def estandarizar_tipos(df, table_name):
    logger.info(f"  🔍 [5/8] ESTANDARIZANDO tipos de datos...")

    tipos = TIPOS_ESTANDAR.get(table_name, {})
    for columna, tipo in tipos.items():
        if columna in df.columns:
            df = df.withColumn(columna, col(columna).cast(tipo))

    # Strings: TRIM + UPPER en categorías
    cols_upper = ["tip_doc", "cod_segmento", "estado_cli", "canal_adquis",
                  "tip_prod", "estado_prod", "tip_mov", "cod_canal",
                  "cod_estado_mov", "calif_riesgo", "tip_comision",
                  "estado_cobro", "tip_punto"]

    for c in cols_upper:
        if c in df.columns:
            df = df.withColumn(c, upper(trim(col(c))))

    logger.info(f"  ✅ Tipos estandarizados")
    return df


# ================================================================
# PASO 6: VALIDAR INTEGRIDAD REFERENCIAL
# Requisito: "los registros que referencien un identificador
#             inexistente deben ser enviados a una tabla de errores
#             con el motivo documentado"
# ================================================================
RELACIONES = {
    "TB_MOV_FINANCIEROS": [
        {"fk": "id_cli", "ref_table": "TB_CLIENTES_CORE", "ref_key": "id_cli"},
        {"fk": "cod_prod", "ref_table": "TB_PRODUCTOS_CAT", "ref_key": "cod_prod"},
    ],
    "TB_OBLIGACIONES": [
        {"fk": "id_cli", "ref_table": "TB_CLIENTES_CORE", "ref_key": "id_cli"},
        {"fk": "cod_prod", "ref_table": "TB_PRODUCTOS_CAT", "ref_key": "cod_prod"},
    ],
    "TB_COMISIONES_LOG": [
        {"fk": "id_cli", "ref_table": "TB_CLIENTES_CORE", "ref_key": "id_cli"},
        {"fk": "cod_prod", "ref_table": "TB_PRODUCTOS_CAT", "ref_key": "cod_prod"},
    ],
}


def validar_integridad_referencial(df, table_name):
    relaciones = RELACIONES.get(table_name, [])
    if not relaciones:
        return df

    logger.info(f"  🔍 [6/8] INTEGRIDAD REFERENCIAL...")

    for rel in relaciones:
        fk = rel["fk"]
        ref_table = rel["ref_table"]
        ref_key = rel["ref_key"]

        # Leer dimensión desde Bronze (ya validada)
        df_ref = spark.sql(f"SELECT DISTINCT {ref_key} FROM {bronze(ref_table)}")

        # LEFT ANTI JOIN: registros que NO tienen match
        df_huerfanos = df.join(
            df_ref,
            df[fk] == df_ref[ref_key],
            "left_anti"
        ).filter(col(fk).isNotNull())

        num_huerfanos = df_huerfanos.count()

        if num_huerfanos > 0:
            logger.info(f"  🔴 {fk} → {ref_table}: {num_huerfanos:,} IDs inexistentes")

            # Enviar a tabla de errores con motivo documentado
            df_huerfanos \
                .withColumn("_error_tipo", lit("INTEGRIDAD_REFERENCIAL")) \
                .withColumn("_error_motivo", lit(f"{fk} no existe en {ref_table}")) \
                .withColumn("_error_fk", col(fk).cast("string")) \
                .withColumn("_fec_deteccion", current_timestamp()) \
                .write.format("delta").mode("append") \
                .option("mergeSchema", "true") \
                .saveAsTable(error_table(table_name))

            # Eliminar huérfanos
            df = df.join(
                df_ref,
                df[fk] == df_ref[ref_key],
                "left_semi"
            )

            error_log.append({
                "tabla": table_name, "tipo": "INTEGRIDAD_REFERENCIAL",
                "motivo": f"{fk} no existe en {ref_table}", "registros": num_huerfanos
            })
        else:
            logger.info(f"  ✅ {fk} → {ref_table}: todos existen")

    return df


# ================================================================
# PASO 7: ESTRATEGIA DOCUMENTADA DE NULOS
# Requisito: "imputación con valor por defecto, exclusión del
#             registro o marcado con un indicador binario"
# ================================================================
#
# Estrategia por columna:
# ┌─────────────────────┬──────────────────┬───────────────────────┐
# │ Columna             │ Estrategia       │ Valor / Acción        │
# ├─────────────────────┼──────────────────┼───────────────────────┤
# │ depto_res            │ DEFAULT          │ "SIN_DEPARTAMENTO"    │
# │ canal_adquis         │ DEFAULT          │ "DESCONOCIDO"         │
# │ ciudad_res           │ DEFAULT          │ "SIN_CIUDAD"          │
# │ cod_segmento         │ DEFAULT          │ "B" (Básico)          │
# │ cod_ciudad           │ DEFAULT          │ "000"                 │
# │ id_dispositivo       │ DEFAULT          │ "DESCONOCIDO"         │
# │ calif_riesgo         │ DEFAULT          │ "E" (peor caso)       │
# │ dias_mora_act        │ DEFAULT          │ 0                     │
# │ num_cuotas_pend      │ DEFAULT          │ 0                     │
# │ depto (sucursales)   │ DEFAULT          │ "SIN_DEPARTAMENTO"    │
# │ tip_comision         │ DEFAULT          │ "DESCONOCIDO"         │
# │ estado_cobro         │ DEFAULT          │ "PENDIENTE"           │
# │ fec_nac              │ FLAG BINARIO     │ _is_null_fec_nac      │
# │ fec_alta             │ FLAG BINARIO     │ _is_null_fec_alta     │
# │ fec_mov              │ FLAG BINARIO     │ _is_null_fec_mov      │
# │ fec_desembolso       │ FLAG BINARIO     │ _is_null_fec_desemb.  │
# │ fec_venc             │ FLAG BINARIO     │ _is_null_fec_venc     │
# │ fec_cobro            │ FLAG BINARIO     │ _is_null_fec_cobro    │
# │ Campos obligatorios  │ EXCLUSIÓN        │ Enviados a ERR_*      │
# └─────────────────────┴──────────────────┴───────────────────────┘

ESTRATEGIA_NULOS = {
    "TB_CLIENTES_CORE": {
        "depto_res": "SIN_DEPARTAMENTO",
        "canal_adquis": "DESCONOCIDO",
        "ciudad_res": "SIN_CIUDAD",
        "cod_segmento": "B",
    },
    "TB_MOV_FINANCIEROS": {
        "cod_ciudad": "000",
        "id_dispositivo": "DESCONOCIDO",
    },
    "TB_OBLIGACIONES": {
        "calif_riesgo": "E",
        "dias_mora_act": 0,
        "num_cuotas_pend": 0,
    },
    "TB_SUCURSALES_RED": {
        "depto": "SIN_DEPARTAMENTO",
    },
    "TB_COMISIONES_LOG": {
        "tip_comision": "DESCONOCIDO",
        "estado_cobro": "PENDIENTE",
    },
}

NULOS_FLAG = {
    "TB_CLIENTES_CORE": ["fec_nac", "fec_alta"],
    "TB_MOV_FINANCIEROS": ["fec_mov"],
    "TB_OBLIGACIONES": ["fec_desembolso", "fec_venc"],
    "TB_COMISIONES_LOG": ["fec_cobro"],
}


def aplicar_estrategia_nulos(df, table_name):
    logger.info(f"  🔍 [7/8] ESTRATEGIA DE NULOS...")

    # A) Imputación con valor por defecto
    defaults = ESTRATEGIA_NULOS.get(table_name, {})
    for columna, valor in defaults.items():
        if columna in df.columns:
            nulos = df.filter(col(columna).isNull()).count()
            if nulos > 0:
                df = df.withColumn(columna, coalesce(col(columna), lit(valor)))
                logger.info(f"    📌 {columna}: {nulos:,} nulos → imputados con '{valor}'")

    # B) Marcado con indicador binario (fechas)
    flags = NULOS_FLAG.get(table_name, [])
    for columna in flags:
        if columna in df.columns:
            nulos = df.filter(col(columna).isNull()).count()
            df = df.withColumn(
                f"_is_null_{columna}",
                when(col(columna).isNull(), lit(1)).otherwise(lit(0))
            )
            if nulos > 0:
                logger.info(f"    🏷️  {columna}: {nulos:,} nulos → marcados con _is_null_{columna}")

    # C) Exclusión: ya se hizo en paso 2 (nulos en campos obligatorios ��� ERR_*)

    logger.info(f"  ✅ Estrategia de nulos aplicada")
    return df


# ================================================================
# PASO 8: ENMASCARAMIENTO PII
# Requisito: "Aplicar enmascaramiento o hash sobre columnas que
#             contengan información de identificación personal"
# ================================================================
COLS_PII = {
    "TB_CLIENTES_CORE": {
        "hash": ["num_doc"],
        "mask": ["nomb_cli", "apell_cli"],
    }
}


def enmascarar_pii(df, table_name):
    pii_config = COLS_PII.get(table_name, {})
    if not pii_config:
        return df

    logger.info(f"  🔍 [8/8] ENMASCARAMIENTO PII...")

    # Hash SHA-256 para documentos de identidad
    for c in pii_config.get("hash", []):
        if c in df.columns:
            df = df.withColumn(f"{c}_hash", sha2(col(c).cast("string"), 256))
            df = df.drop(c)
            logger.info(f"    🔑 {c} → SHA-256 (columna original eliminada, nueva: {c}_hash)")

    # Enmascaramiento parcial para nombres: "Juan" → "J***"
    for c in pii_config.get("mask", []):
        if c in df.columns:
            df = df.withColumn(
                c,
                when(
                    col(c).isNotNull(),
                    concat_ws("", col(c).substr(1, 1), lit("***"))
                ).otherwise(lit("***"))
            )
            logger.info(f"    🎭 {c} → enmascarado (primera letra + ***)")

    logger.info(f"  ✅ PII protegida")
    return df

# ================================================================
# PASO 9: DETECCIÓN DE TRANSACCIONES SOSPECHOSAS
# Requisito: "Marcar una transacción como sospechosa cuando su
#             vr_mov supere en más de 3 desviaciones estándar el
#             promedio de los últimos 30 días del mismo cliente.
#             El flag ind_sospechoso debe calcularse en Silver
#             antes de llegar a Gold."
# ================================================================
def detectar_sospechosas(df, table_name):
    if table_name != "TB_MOV_FINANCIEROS":
        return df

    logger.info(f"  🔍 [9/9] DETECCIÓN DE TRANSACCIONES SOSPECHOSAS...")

    w_cliente = Window.partitionBy("id_cli").orderBy("fec_mov").rowsBetween(-30, -1)

    df = df.withColumn("prom_movil_30d", spark_round(avg("vr_mov").over(w_cliente), 2))
    df = df.withColumn("stddev_movil_30d", spark_round(stddev("vr_mov").over(w_cliente), 2))

    df = df.withColumn(
        "ind_sospechoso",
        when(
            (col("prom_movil_30d").isNotNull()) &
            (col("stddev_movil_30d").isNotNull()) &
            (col("stddev_movil_30d") > 0) &
            (col("vr_mov") > col("prom_movil_30d") + lit(3) * col("stddev_movil_30d")),
            lit(True)
        ).otherwise(lit(False))
    )

    sosp = df.filter(col("ind_sospechoso") == True).count()
    logger.info(f"  🔴 {sosp:,} transacciones sospechosas detectadas")

    return df

# ================================================================
# PROCESAR CADA TABLA (los 9 pasos en orden)
# ================================================================
def procesar_tabla(table_name, config):
    source = bronze(table_name)
    target = silver(table_name)
    pk = config["primary_key"]

    logger.info(f"\n{'=' * 70}")
    logger.info(f"📋 {table_name}")
    logger.info(f"   📥 Origen:  {source}")
    logger.info(f"   📤 Destino: {target}")
    logger.info(f"{'=' * 70}")

    t0 = time.time()

    # Limpiar tabla de errores anterior
    try:
        spark.sql(f"DROP TABLE IF EXISTS {error_table(table_name)}")
    except:
        pass

    df = spark.sql(f"SELECT * FROM {source}")
    total_original = df.count()
    logger.info(f"  📊 Total registros origen: {total_original:,}")

    # Ejecutar los 9 pasos en orden
    df = eliminar_duplicados(df, table_name, pk)                       # Paso 1
    df = eliminar_nulos_criticos(df, table_name, config["non_null_cols"])  # Paso 2
    df = detectar_fechas_anomalas(df, table_name, config["date_cols"])    # Paso 3
    df = corregir_inconsistencias(df, table_name)                        # Paso 4
    df = estandarizar_tipos(df, table_name)                              # Paso 5
    df = validar_integridad_referencial(df, table_name)                  # Paso 6
    df = aplicar_estrategia_nulos(df, table_name)                        # Paso 7
    df = enmascarar_pii(df, table_name)                                  # Paso 8
    df = detectar_sospechosas(df, table_name)                             # Paso 9 

    # Guardar en Silver
    total_final = df.count()
    df.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target)

    duration = time.time() - t0
    rechazados = total_original - total_final
    pct = round((total_final / total_original * 100), 2) if total_original > 0 else 0

    logger.info(f"\n  💾 Guardado: {target}")
    logger.info(f"  📊 Origen: {total_original:,} → Silver: {total_final:,} → Rechazados: {rechazados:,}")
    logger.info(f"  📈 % Conformes: {pct}%")
    logger.info(f"  ⏱️  {duration:.2f}s")

    # Generar reporte de calidad para esta tabla
    generar_reporte_calidad(df, table_name, total_original, total_final)


# ================================================================
# REPORTE DE CALIDAD DE DATOS
# Requisito: "porcentaje de nulos por columna, número de registros
#             rechazados y porcentaje de registros conformes"
# ================================================================
def generar_reporte_calidad(df, table_name, total_original, total_final):
    rechazados = total_original - total_final
    pct_conformes = round((total_final / total_original * 100), 2) if total_original > 0 else 0

    # % nulos por columna (solo columnas de datos, no flags)
    nulos_por_columna = {}
    columnas_datos = [c for c in df.columns if not c.startswith("_")]
    for c in columnas_datos:
        cnt_nulo = df.filter(col(c).isNull()).count()
        pct_nulo = round((cnt_nulo / total_final * 100), 2) if total_final > 0 else 0
        nulos_por_columna[c] = {"count": cnt_nulo, "pct": pct_nulo}

    errores_tabla = [e for e in error_log if e["tabla"] == table_name]

    quality_report.append({
        "tabla": table_name,
        "registros_origen": total_original,
        "registros_silver": total_final,
        "registros_rechazados": rechazados,
        "pct_conformes": pct_conformes,
        "nulos_por_columna": nulos_por_columna,
        "errores": errores_tabla,
    })


# ================================================================
# IMPRIMIR REPORTE DE CALIDAD
# ================================================================
def imprimir_reporte():
    logger.info(f"\n{'=' * 70}")
    logger.info("📊 REPORTE DE CALIDAD DE DATOS — SILVER")
    logger.info(f"{'=' * 70}")

    # --- Resumen por tabla ---
    logger.info(f"\n  {'Tabla':<25} {'Origen':>10} {'Silver':>10} {'Rechaz.':>10} {'% Conf.':>8}")
    logger.info(f"  {'─' * 65}")

    for r in quality_report:
        logger.info(
            f"  {r['tabla']:<25} {r['registros_origen']:>10,} "
            f"{r['registros_silver']:>10,} {r['registros_rechazados']:>10,} "
            f"{r['pct_conformes']:>7}%"
        )

    # --- Totales ---
    total_origen = sum(r['registros_origen'] for r in quality_report)
    total_silver = sum(r['registros_silver'] for r in quality_report)
    total_rechazados = sum(r['registros_rechazados'] for r in quality_report)
    pct_global = round((total_silver / total_origen * 100), 2) if total_origen > 0 else 0

    logger.info(f"  {'─' * 65}")
    logger.info(
        f"  {'TOTAL':<25} {total_origen:>10,} "
        f"{total_silver:>10,} {total_rechazados:>10,} "
        f"{pct_global:>7}%"
    )

    # --- Nulos por columna ---
    logger.info(f"\n  📋 NULOS POR COLUMNA (solo donde hay nulos):")
    logger.info(f"  {'─' * 65}")
    for r in quality_report:
        nulos_con_datos = {k: v for k, v in r['nulos_por_columna'].items() if v['count'] > 0}
        if nulos_con_datos:
            logger.info(f"  {r['tabla']}:")
            for col_name, info in nulos_con_datos.items():
                logger.info(f"    • {col_name}: {info['count']:,} ({info['pct']}%)")

    # --- Errores documentados ---
    if error_log:
        logger.info(f"\n  📋 ERRORES DOCUMENTADOS:")
        logger.info(f"  {'─' * 65}")
        logger.info(f"  {'Tabla':<25} {'Tipo':<25} {'Motivo':<30} {'Reg.':>8}")
        logger.info(f"  {'─' * 65}")
        for e in error_log:
            logger.info(
                f"  {e['tabla']:<25} {e['tipo']:<25} "
                f"{e['motivo']:<30} {e['registros']:>8,}"
            )

    # --- Anomalías de Fase 1 detectadas ---
    if anomalias_detectadas:
        logger.info(f"\n  🔴 ANOMALÍAS DE FASE 1 DETECTADAS:")
        logger.info(f"  {'─' * 65}")
        for a in anomalias_detectadas:
            logger.info(
                f"  ✅ {a['anomalia']}: {a['descripcion']}"
                f" → {a['tabla']} ({a['detectados']:,} registros)"
            )

    logger.info(f"\n{'=' * 70}")

# ================================================================
# GUARDAR REPORTE DE CALIDAD COMO TABLA
# ================================================================
def guardar_reporte_calidad():
    """
    Genera la tabla slv_reporte_calidad con:
    - % nulos por columna
    - Numero de registros rechazados
    - % registros conformes
    """
    from pyspark.sql import Row

    filas = []

    for r in quality_report:
        # Fila resumen de la tabla
        filas.append(Row(
            tabla=r["tabla"],
            columna="_RESUMEN_TABLA_",
            registros_origen=r["registros_origen"],
            registros_silver=r["registros_silver"],
            registros_rechazados=r["registros_rechazados"],
            pct_conformes=r["pct_conformes"],
            nulos_count=0,
            nulos_pct=0.0
        ))

        # Una fila por cada columna con su % de nulos
        for col_name, info in r["nulos_por_columna"].items():
            filas.append(Row(
                tabla=r["tabla"],
                columna=col_name,
                registros_origen=r["registros_origen"],
                registros_silver=r["registros_silver"],
                registros_rechazados=r["registros_rechazados"],
                pct_conformes=r["pct_conformes"],
                nulos_count=info["count"],
                nulos_pct=info["pct"]
            ))

    df_reporte = spark.createDataFrame(filas)

    target = f"`{CATALOG}`.{SILVER_LH}.{SILVER_SCHEMA}.slv_reporte_calidad"
    # Obtener timestamp local (según la sesión) y guardarlo como UTC + string local legible
    ingest_ts_local = current_timestamp()  # asume spark.conf.set("spark.sql.session.timeZone","America/Bogota") ya configurado
    ingest_ts_utc = to_utc_timestamp(ingest_ts_local, "America/Bogota")
    ingest_ts_local_str = date_format(ingest_ts_local, "yyyy-MM-dd HH:mm:ss")


    df_reporte = (
        df_reporte
        .withColumn("_fec_ejecucion", ingest_ts_local_str)           # timestamp UTC (oficial)
    )

    df_reporte.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target)

    logger.info(f"\n  📊 Reporte guardado en: {target}")
    logger.info(f"     {len(filas)} filas (resumen + detalle por columna)")


# ================================================================
# EJECUCIÓN
# ================================================================
if __name__ == "__main__":
    t0 = time.time()

    logger.info(f"\n{'=' * 70}")
    logger.info("🥈 PIPELINE SILVER — LIMPIEZA Y CONFORMACIÓN")
    logger.info(f"   Origen:  BRZ_* (Bronze)")
    logger.info(f"   Destino: SLV_* (Silver) + ERR_* (Errores)")
    logger.info(f"{'=' * 70}")
    logger.info(f"""
    Pasos que ejecuta:
      1. Eliminar duplicados (Anomalía 1 - Fase 1)
      2. Eliminar nulos en campos obligatorios
      3. Detectar fechas fuera de rango (Anomalía 2 - Fase 1)
      4. Corregir inconsistencias lógicas (Anomalía 3 - Fase 1)
      5. Estandarizar tipos de datos
      6. Validar integridad referencial → ERR_*
      7. Aplicar estrategia documentada de nulos
      8. Enmascarar datos personales (PII)
      9. Detección de transacciones sospechosas 
    """)

    # Procesar TODAS las tablas
    for table_name, config in TABLES_CONFIG.items():

        def tarea():
            procesar_tabla(table_name, config)

        ejecutar_con_manejo_errores(
            nombre_tarea=f"Silver_{table_name}",
            funcion=tarea
        )

    # Reporte de calidad
    imprimir_reporte()
    guardar_reporte_calidad() 
    guardar_errores_pipeline()

    duration = time.time() - t0
    logger.info(f"\n⏱️  Duración total: {duration:.2f}s")
    logger.info("✅ Pipeline Silver completado")

StatementMeta(, 53313e9a-d035-4285-982c-d89b1beaa476, 5, Finished, Available, Finished, False)

2026-04-07 16:13:18,682 - INFO - 
2026-04-07 16:13:18,683 - INFO - 🥈 PIPELINE SILVER — LIMPIEZA Y CONFORMACIÓN
2026-04-07 16:13:18,683 - INFO -    Origen:  BRZ_* (Bronze)
2026-04-07 16:13:18,684 - INFO -    Destino: SLV_* (Silver) + ERR_* (Errores)
2026-04-07 16:13:18,684 - INFO - ======================================================================
2026-04-07 16:13:18,685 - INFO - 
    Pasos que ejecuta:
      1. Eliminar duplicados (Anomalía 1 - Fase 1)
      2. Eliminar nulos en campos obligatorios
      3. Detectar fechas fuera de rango (Anomalía 2 - Fase 1)
      4. Corregir inconsistencias lógicas (Anomalía 3 - Fase 1)
      5. Estandarizar tipos de datos
      6. Validar integridad referencial → ERR_*
      7. Aplicar estrategia documentada de nulos
      8. Enmascarar datos personales (PII)
      9. Detección de transacciones sospechosas 
    
2026-04-07 16:13:18,685 - INFO - 
2026-04-07 16:13:18,686 - INFO - 📋 TB_CLIENTES_CORE
2026-04-07 16:13:18,686 - INFO -    📥 Origen:  `P